# DATA PREPARATION

In [1]:
# ------------------------------------------
# INITIATION: PYTHON AND DISPLAY
# ------------------------------------------

# Import libraries
import pandas as pd #
import numpy as np
import ipywidgets as widgets
import matplotlib.pyplot as plt #charts
import seaborn as sns #charts
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.stats import shapiro, jarque_bera

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

from statsmodels.regression.quantile_regression import QuantReg
import statsmodels.formula.api as smf

import shap

from sklearn.linear_model import LassoCV



# For better visibility
from IPython.display import display, HTML

# Apply a no-wrap style to prevent text wrapping
display(HTML("""
<style>
.dataframe td { white-space: nowrap; }
</style>
"""))

# Set the option to display all columns
pd.set_option('display.max_columns', None)

# Set the option to display all rows
pd.set_option('display.max_rows', None)

# Set the float format to use a space as a thousands separator and a comma as a decimal separator
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' ').replace('.', ','))

In [2]:
# ------------------------------------------
# FILE LOAD
# ------------------------------------------

# Read the Excel file into a DataFrame
df = pd.read_excel('2000 Analysis Python CHANGED.xlsx', decimal=',')
display (df.head())

,2023,2022,2021,2020,2019,Name,NIP,PKD,PKD Opis,Sector,Owner Type,Year,Sales,Gross Profit,Operating Profit,Net Profit,Depreciation,Export Sales,Employment,Staff costs,BS,Fixed Assets,Current Assets,Equity,Liabilities,PKD_2,PKD_1,PKD_1 Description,PKD_2 Description,Quartile of digital intensity: 2001-03,Quartile of digital intensity: 2013-15,Export as % of Sales,ROS,ROA,ROE,P/F,BS_Strength,Human_Capital,GVA,Productivity,Hint25,Hint50,HGX,HGX%,SGrowth_2023_2019,LSGrowth_2023_2019,RSGrowth_2023_2019,LRSGrowth_2023_2019,Annualised_Growth,Drop_Sales_2020,Drop_Sales_2023,Drop_Sales_Both,SGrowth_3,SGrowth_2,SGrowth_1,GVAGrowth_2023_2019,EGrowth_2023_2019,LEGrowth_2023_2019,Annualised_EGrowth,Drop_Employment,EGrowth_3,EGrowth_2,PGrowth_2023_2019,Starting_Profit,Starting_INT,Starting_BS_Strength,Starting_Size_arc,Starting_Size_ln,Starting_Human_Capital,Starting_GVA,Starting_Prod
0,2,"2,00","3,00","2,00",2,"Jeronimo Martins Polska SA, Kostrzyn",7791011327,4719,Pozostała sprzedaż detaliczna prowadzona w nie...,Retail,500,2023,"98 022 928,00","4 882 170,96","5 121 150,49","4 127 757,73","1 311 227,50","0,00","79 292,00","5 954 379,19","26 597 526,53","11 978 435,65","14 619 090,88","5 299 163,00","21 298 363,10",47,G,Wholesale and Retail Trade; Repair of Motor Ve...,"RETAIL TRADE, EXCEPT OF MOTOR VEHICLES",Medium-high,Medium-high,"0,00","0,04","0,16","0,78",F,"0,20","75,09","10 836 550,15","136,67",Exp<25,Exp<50,Decreased Export Intensity,"0,00","76,38","0,57","27,81","0,25","15,24",No,No,No,1.Moderate,2.Stable,Growing,"4 394 364,76","16,33","0,15","3,85",No,2.Hiring,Hiring,"42,15","0,04","0,00","0,23","18,53","17,83","53,43","6 442 185,39","94,52"
1,3,"3,00","4,00","3,00",4,"PGE Polska Grupa Energetyczna GK SA, Lublin",5260250541,7010,Działalność firm centralnych (head offices) i ...,Energy,145,2023,"95 964 000,00","-3 431 000,00","-4 055 000,00","-4 902 000,00","13 455 000,00","25 000,00","42 552,00","6 287 000,00","113 443 000,00","78 340 000,00","35 103 000,00","47 855 000,00","65 588 000,00",70,M,"Professional, Scientific and Technical Activities",ACTIVITIES OF HEAD OFFICES; MANAGEMENT CONSULT...,High,High,"0,00","-0,05","-0,04","-0,10",P,"0,42","147,75","2 856 000,00","67,12",Exp<25,Exp<50,Decreased Export Intensity,"-0,00","155,04","0,94","84,81","0,61","26,37",No,No,No,2.High,3.Growing,Growing,"2 373 000,00","1,47","0,01","0,37",Yes,1.Sustaining,Hiring,"55,60","-0,10","0,00","0,56","18,14","17,44","111,08","483 000,00","11,52"
2,4,"5,00","11,00","17,00",30,"Totalizator Sportowy sp. z o.o. GK, Warszawa",5250010982,9200,Działalność związana z grami losowymi i zakład...,Services,100,2023,"52 269 058,00","385 142,00","453 419,00","362 474,00","189 397,00","0,00","5 894,00","786 162,00","2 893 324,00","1 099 674,00","1 793 650,00","1 669 328,00","1 223 996,00",92,R,"Arts, Entertainment and Recreation",GAMBLING AND BETTING ACTIVITIES,Medium-low,Medium-high,"0,00","0,01","0,13","0,22",P,"0,58","133,38","1 171 304,00","198,73",Exp<25,Exp<50,Decreased Export Intensity,"0,00","428,35","1,66","282,86","1,34","51,61",No,No,No,3.Very High,3.Growing,Growing,"513 096,67","221,37","1,17","33,89",No,2.Hiring,Hiring,"-160,16","0,03","0,00","0,59","16,80","16,11","188,29","658 207,33","358,89"
3,5,"11,00","14,00","13,00",15,"Enea SA GK, Poznań",7770020640,3514,Handel energią elektryczną,Energy,145,2023,"48 183 419,00","955 679,00","-508 049,00","-442 623,00","1 651 920,00","0,00","17 853,00","3 092 964,00","39 110 745,00","21 636 978,00","17 473 767,00","15 439 599,00","23 671 146,00",35,D,"Electricity, Gas, Steam, Hot Water and Air Con...","ELECTRICITY, GAS, STEAM, HOT WATER AND AIR CON...",Low,Low,"0,00","-0,01","-0,01","-0,03",P,"0,39","173,25","4 048 643,00","226,78",Exp<25,Exp<50,Decreased Export Intensity,"0,00","205,03","1,12","121,04","0,79","32,16",No,No,No,3.Very High,3.Growing,Growing,"360 494,00","5,04","0,05","1,24",No,2.Hiring,Hiring,"9,78","0,03","0,00","0,47","17,27","16,58","112,03","3 688 149,00","217,00"
4,6,"6,00","13,00","11,00"

In [3]:
# ------------------------------------------
# VARIABLE MANAGEMENT TEMPLATE
# ------------------------------------------

# 1. Load your full dataset (already cleaned)
df_master = df

# 🟨 ID & Metadata (for exclusion in all analysis)
id_columns = ['NIP','PKD', 'PKD_1 Description', 'PKD_2 Description','2019', '2020', '2021', '2022', '2023']

# 🟥 Outcome-derived (Exclude from modeling but use in descriptive)
derived_outcomes = [
    'Annualised_Growth', 'Annualised_EGrowth', 'SGrowth_2023_2019','EGrowth_2023_2019','GVAGrowth_2023_2019',
    'PGrowth_2023_2019', '3_Growth_Categories','4_Growth_Categories',
    'EGrowth_Category','Hint25','Hint50'
]

# ✅ Modeling variables (filtered dynamically)
all_exclusions = id_columns + derived_outcomes

# 3. Build filtered dataset for modeling
df_model = df_master.copy()
df_model = df_model[df_model["Year"] == 2023]  # Example: focus on 2023
df_model = df_model.drop(columns=[col for col in all_exclusions if col in df_model.columns])

display(df_model.head())

,Name,PKD Opis,Sector,Owner Type,Year,Sales,Gross Profit,Operating Profit,Net Profit,Depreciation,Export Sales,Employment,Staff costs,BS,Fixed Assets,Current Assets,Equity,Liabilities,PKD_2,PKD_1,Quartile of digital intensity: 2001-03,Quartile of digital intensity: 2013-15,Export as % of Sales,ROS,ROA,ROE,P/F,BS_Strength,Human_Capital,GVA,Productivity,HGX,HGX%,LSGrowth_2023_2019,RSGrowth_2023_2019,LRSGrowth_2023_2019,Drop_Sales_2020,Drop_Sales_2023,Drop_Sales_Both,SGrowth_3,SGrowth_2,SGrowth_1,LEGrowth_2023_2019,Drop_Employment,EGrowth_3,EGrowth_2,Starting_Profit,Starting_INT,Starting_BS_Strength,Starting_Size_arc,Starting_Size_ln,Starting_Human_Capital,Starting_GVA,Starting_Prod
0,"Jeronimo Martins Polska SA, Kostrzyn",Pozostała sprzedaż detaliczna prowadzona w nie...,Retail,500,2023,"98 022 928,00","4 882 170,96","5 121 150,49","4 127 757,73","1 311 227,50","0,00","79 292,00","5 954 379,19","26 597 526,53","11 978 435,65","14 619 090,88","5 299 163,00","21 298 363,10",47,G,Medium-high,Medium-high,"0,00","0,04","0,16","0,78",F,"0,20","75,09","10 836 550,15","136,67",Decreased Export Intensity,"0,00","0,57","27,81","0,25",No,No,No,1.Moderate,2.Stable,Growing,"0,15",No,2.Hiring,Hiring,"0,04","0,00","0,23","18,53","17,83","53,43","6 442 185,39","94,52"
1,"PGE Polska Grupa Energetyczna GK SA, Lublin",Działalność firm centralnych (head offices) i ...,Energy,145,2023,"95 964 000,00","-3 431 000,00","-4 055 000,00","-4 902 000,00","13 455 000,00","25 000,00","42 552,00","6 287 000,00","113 443 000,00","78 340 000,00","35 103 000,00","47 855 000,00","65 588 000,00",70,M,High,High,"0,00","-0,05","-0,04","-0,10",P,"0,42","147,75","2 856 000,00","67,12",Decreased Export Intensity,"-0,00","0,94","84,81","0,61",No,No,No,2.High,3.Growing,Growing,"0,01",Yes,1.Sustaining,Hiring,"-0,10","0,00","0,56","18,14","17,44","111,08","483 000,00","11,52"
2,"Totalizator Sportowy sp. z o.o. GK, Warszawa",Działalność związana z grami losowymi i zakład...,Services,100,2023,"52 269 058,00","385 142,00","453 419,00","362 474,00","189 397,00","0,00","5 894,00","786 162,00","2 893 324,00","1 099 674,00","1 793 650,00","1 669 328,00","1 223 996,00",92,R,Medium-low,Medium-high,"0,00","0,01","0,13","0,22",P,"0,58","133,38","1 171 304,00","198,73",Decreased Export Intensity,"0,00","1,66","282,86","1,34",No,No,No,3.Very High,3.Growing,Growing,"1,17",No,2.Hiring,Hiring,"0,03","0,00","0,59","16,80","16,11","188,29","658 207,33","358,89"
3,"Enea SA GK, Poznań",Handel energią elektryczną,Energy,145,2023,"48 183 419,00","955 679,00","-508 049,00","-442 623,00","1 651 920,00","0,00","17 853,00","3 092 964,00","39 110 745,00","21 636 978,00","17 473 767,00","15 439 599,00","23 671 146,00",35,D,Low,Low,"0,00","-0,01","-0,01","-0,03",P,"0,39","173,25","4 048 643,00","226,78",Decreased Export Intensity,"0,00","1,12","121,04","0,79",No,No,No,3.Very High,3.Growing,Growing,"0,05",No,2.Hiring,Hiring,"0,03","0,00","0,47","17,27","16,58","112,03","3 688 149,00","217,00"
4,"Tauron Polska Energia SA GK, Katowice",Handel energią elektryczną,Energy,145,2023,"42 657 000,00","3 394 000,00","2 302 000,00","1 678 000,00","2 226 000,00","753 000,00","18 946,00","2 851 000,00","49 798 000,00","37 353 000,00","12 445 000,00","17 953 000,00","31 845 000,00",35,D,Low,Low,"0,02","0,04","0,03","0,09",P,"0,36","150,48","6 245 000,00","329,62",Increased Export Intensity,"0,01","0,80","61,73","0,48",No,No,No,2.High,3.Growing,Growing,"-0,32",Yes,0.Reducing,Reducing,"-0,00","0,01","0,46","17,46","16,77","98,27","2 762 520,00","105,90"


In [4]:
# ------------------------------------------
# DATA PREPARATION
# ------------------------------------------

# STEP 1: FILTER TO YEAR 2023
df_2023 = df_model[df_model['Year'] == 2023]

# STEP 2: DEFINE VARIABLES
target_variable = 'LSGrowth_2023_2019'
numeric_features = ['Starting_INT', 'Starting_BS_Strength', 'Starting_Size_ln', 'Starting_Human_Capital', 'Starting_Profit', 'Starting_Prod']
categorical_vars = ['P/F', 'Sector']
winsor_percentile = 0.01  # Set to 0 to disable winsorization

# STEP 3: DROP MISSING VALUES (WITH COPY)
all_vars = numeric_features + categorical_vars + [target_variable]
before_drop = df_2023["Name"].tolist()

# Create a full copy to avoid chained assignment issues
df_clean = df_2023.dropna(subset=all_vars).copy()
after_drop = df_clean["Name"].tolist()

# Identify dropped companies
dropped_companies = set(before_drop) - set(after_drop)
dropped_count = len(dropped_companies)

print(f"\nCompanies dropped due to missing values (Total: {dropped_count}):\n")
print("\n".join(sorted(dropped_companies)))

# STEP 4: CONDITIONAL WINSORISATION
if winsor_percentile > 0:
    # Calculate lower and upper bounds for winsorization
    lower = df_clean[target_variable].quantile(winsor_percentile)
    upper = df_clean[target_variable].quantile(1 - winsor_percentile)

    # Print distribution before winsorization
    print("\nBefore Winsorization:")
    print(df_clean[target_variable].describe(percentiles=[winsor_percentile, 0.25, 0.5, 0.75, 1 - winsor_percentile]))

    # Create flag for winsorized values (corrected logic)
    flag_col = f"{target_variable}_was_winsorized"
    df_clean.loc[:, flag_col] = (df_clean[target_variable] < lower) | (df_clean[target_variable] > upper)

    # Create the winsorized column
    win_col = f"{target_variable}_win"
    df_clean.loc[:, win_col] = np.clip(df_clean[target_variable], lower, upper)

    # Update target variable reference for regression block
    target_variable = win_col  

    # Print distribution after winsorization
    print("\nAfter Winsorization:")
    print(df_clean[target_variable].describe(percentiles=[winsor_percentile, 0.25, 0.5, 0.75, 1 - winsor_percentile]))

    # Count and log winsorized companies
    winsorized_count = df_clean[flag_col].sum()
    print(f"\nTotal number of companies with winsorized '{target_variable}': {winsorized_count}\n")
else:
    print("\nWinsorization skipped as percentile is set to 0.\n")

# STEP 5: SAVE CLEANED DATA
df_clean.to_excel("2000 Analysis Python WINSOR.xlsx", index=False)
print("\nCleaned DataFrame saved to '2000 Analysis Python WINSOR.xlsx'.")



Companies dropped due to missing values (Total: 14):

Grupa Muszkieterów (Intermarché, Bricomarché), Swadzim
Grupa Strabag w Polsce, Pruszków
Intermeble sp. z o.o. sp.k., Poznań
KPMG sp. z o.o. GK, Warszawa 
Kuehne + Nagel sp. z o.o., Gądki
La Lorraine Polska sp. z o.o., Nowy Dwór Mazowiecki
PSH Lewiatan GK, Włocławek
Pilkington (NSG Group w Polsce), Sandomierz
Polmlek sp. z o.o. GK, Warszawa
Polska Żegluga Morska PP GK, Szczecin
PwC sp. z o.o. GK, Warszawa 
Suempol sp. z o.o., Bielsk Podlaski
Super-Pharm Poland sp. z o.o., Warszawa
ZPUE SA, Włoszczowa

Before Winsorization:
count   1 489,00
mean        0,40
std         0,36
min        -1,28
1%         -0,42
25%         0,20
50%         0,37
75%         0,55
99%         1,40
max         5,87
Name: LSGrowth_2023_2019, dtype: float64

After Winsorization:
count   1 489,00
mean        0,39
std         0,31
min        -0,42
1%         -0,42
25%         0,20
50%         0,37
75%         0,55
99%         1,40
max         1,40
Name: LSGrowth

# OLS Regression

In [5]:
# ------------------------------------------
# OLS REGRESSION ANALYSIS
# ------------------------------------------

# STEP 4: ENCODE CATEGORICAL VARIABLES (drop_first to avoid dummy trap)
X_categorical = pd.get_dummies(df_clean[categorical_vars], drop_first=True)

# Print to verify the available dummy columns
print("\n✅ Available Dummy Variables:")
print(X_categorical.columns)

# STEP 5: STANDARDIZE NUMERIC VARIABLES
X_numeric = pd.DataFrame(
    StandardScaler().fit_transform(df_clean[numeric_features]),
    columns=numeric_features,
    index=df_clean.index
)

# Add interaction terms if specified
for term1, term2 in interaction_terms:
    # Check both possible sources for terms
    if term1 in X_numeric.columns and term2 in X_categorical.columns:
        interaction_name = f"{term1}_Mult_{term2}"
        X_categorical[interaction_name] = X_numeric[term1] * X_categorical[term2]
        print(f"✅ Interaction term added: {interaction_name}")
    elif term2 in X_categorical.columns and term1 in X_categorical.columns:
        interaction_name = f"{term2}_Mult_{term1}"
        X_categorical[interaction_name] = X_categorical[term2] * X_categorical[term1]
        print(f"✅ Interaction term added: {interaction_name}")
    else:
        print(f"⚠️ Warning: Interaction term '{term1} * {term2}' not added (variables not found).")

# STEP 6: COMBINE FEATURES AND ADD CONSTANT
X_full = pd.concat([X_numeric, X_categorical], axis=1)
X_full = sm.add_constant(X_full).astype(float)  # Ensure numeric

# STEP 7: DEFINE TARGET
y = df_clean[target_variable]

# STEP 8: FIT OLS REGRESSION MODEL
model = sm.OLS(y, X_full).fit()

# Extract coefficients as dataframe
coeff_df = pd.DataFrame({
    "Variable": model.params.index,
    "Coef": model.params.values,
    "Std Err": model.bse.values,
    "t": model.tvalues.values,
    "P>|t|": model.pvalues.values,
    "[0.025": model.conf_int()[0].values,
    "0.975]": model.conf_int()[1].values
})

# Display the full model summary
display(model.summary())

# STEP 8B: EXTRACT REGRESSION SUMMARY STATISTICS
summary_stats = {
    "Statistic": ["R-squared", "Adj. R-squared", "F-statistic", "Prob (F-statistic)", 
                  "Log-Likelihood", "AIC", "BIC", "No. Observations", "DF Residuals", "DF Model"],
    "Value": [
        model.rsquared,
        model.rsquared_adj,
        model.fvalue,
        model.f_pvalue,
        model.llf,
        model.aic,
        model.bic,
        model.nobs,
        model.df_resid,
        model.df_model
    ]
}

summary_df = pd.DataFrame(summary_stats)

# Run statistical tests on residuals
residuals = model.resid.replace([np.inf, -np.inf], np.nan).dropna()
shapiro_test = shapiro(residuals)
jb_test = jarque_bera(residuals)

residual_stats = pd.DataFrame({
    "Test": ["Shapiro-Wilk", "Jarque-Bera"],
    "Statistic": [shapiro_test.statistic, jb_test.statistic],
    "p-value": [shapiro_test.pvalue, jb_test.pvalue]
})

# Display results
print("\n🔍 Shapiro-Wilk Test (normality):")
print(f"  Statistic = {shapiro_test.statistic:.4f}")
print(f"  p-value   = {shapiro_test.pvalue:.4f}")

print("\n📦 Jarque-Bera Test (skew + kurtosis):")
print(f"  Statistic = {jb_test.statistic:.4f}")
print(f"  p-value   = {jb_test.pvalue:.4f}")

# STEP 9: CALCULATE VIF TO CHECK MULTICOLLINEARITY
vif_data = pd.DataFrame({
    "Variable": X_full.columns,
    "VIF": [variance_inflation_factor(X_full.values, i) for i in range(X_full.shape[1])]
})
display(vif_data)

# STEP 10: CLEAN RESIDUALS
residuals = model.resid.replace([np.inf, -np.inf], np.nan).dropna()

# STEP 11: RESIDUALS DISTRIBUTION PLOT
sns.histplot(residuals, kde=True)
plt.title("Residuals Distribution")
plt.xlabel("Residuals")
plt.ylabel("Frequency")
plt.show()

# STEP 12: QQ PLOT
sm.qqplot(residuals, line='45')
plt.title("QQ Plot of Residuals")
plt.show()
    
# EXPORT TO EXCEL
with pd.ExcelWriter("Regression_Results.xlsx") as writer:
    summary_df.to_excel(writer, sheet_name="Model_Summary", index=False)
    coeff_df.to_excel(writer, sheet_name="Model_Coefficients", index=False)
    vif_data.to_excel(writer, sheet_name="VIF", index=False)
    residual_stats.to_excel(writer, sheet_name="Residual_Tests", index=False)

print("✅ Results exported to 'Regression_Results.xlsx'")



✅ Available Dummy Variables:
Index(['P/F_P', 'Sector_Chemicals', 'Sector_Construction', 'Sector_Energy',
       'Sector_Food', 'Sector_Healthcare and pharma',
       'Sector_Media, telco and IT', 'Sector_Mining and metallurgy',
       'Sector_Oil', 'Sector_Production', 'Sector_Retail', 'Sector_Services',
       'Sector_Transportation', 'Sector_Wholesale'],
      dtype='object')


NameError: name 'interaction_terms' is not defined

In [6]:
# Exclude "Sector" from the list of categorical variables
categorical_vars_filtered = [var for var in categorical_vars if var != 'Sector']

# Combine numeric and filtered categorical variables
df_combined = pd.get_dummies(
    df_clean[numeric_features + categorical_vars_filtered + [target_variable]],
    drop_first=True
)
# Move the target column to the first position
cols = list(df_combined.columns)
if target in cols:
    cols.insert(0, cols.pop(cols.index(target)))
df_combined = df_combined[cols]

# Calculate the correlation matrix
correlation_matrix = df_combined.corr()

# Reorder the rows as well to place target at the top
correlation_matrix = correlation_matrix.reindex(index=cols, columns=cols)

# Display the correlation matrix
display(correlation_matrix)

# Flatten the correlation matrix and remove self-correlations
correlation_pairs = correlation_matrix.unstack().reset_index()
correlation_pairs.columns = ["Variable 1", "Variable 2", "Correlation"]

# Remove duplicate pairs (A-B and B-A)
correlation_pairs["Pair"] = correlation_pairs.apply(
    lambda row: "-".join(sorted([row["Variable 1"], row["Variable 2"]])), axis=1
)
correlation_pairs.drop_duplicates(subset="Pair", inplace=True)
correlation_pairs.drop(columns=["Pair"], inplace=True)

# Remove self-correlations
correlation_pairs = correlation_pairs[correlation_pairs["Variable 1"] != correlation_pairs["Variable 2"]]

# Sort by absolute correlation value
top_10_correlations = correlation_pairs.reindex(
    correlation_pairs.Correlation.abs().sort_values(ascending=False).index
).head(10)


display(top_10_correlations)

NameError: name 'target' is not defined

In [7]:
df_clean[['Starting_GVA', 'Starting_Profit']].corr()

,Starting_GVA,Starting_Profit
Starting_GVA,"1,00","0,11"
Starting_Profit,"0,11","1,00"


In [8]:
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from sklearn.metrics import r2_score, mean_squared_error

# STEP A: Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_full, y, test_size=0.3, random_state=42)

# STEP B: Fit OLS Model on Training Data
model_train = sm.OLS(y_train, X_train).fit()

# STEP C: Make Predictions on Testing Data
y_test_pred = model_train.predict(X_test)

# STEP D: Evaluate Model Performance
test_r2 = r2_score(y_test, y_test_pred)
test_mse = mean_squared_error(y_test, y_test_pred)

# Display Test Performance
print(f"Test R² Score: {test_r2:.4f}")
print(f"Test Mean Squared Error: {test_mse:.4f}")


NameError: name 'X_full' is not defined

 # Random Forest Feature and Permutations Importance

In [9]:
# 1. Drop 'const' and define input/output
X_rf = X_full.drop(columns='const')
y_rf = y

# 2. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X_rf, y_rf, test_size=0.3, random_state=42)

# 3. Fit Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# 4. Random Forest Feature Importance
rf_importances = pd.Series(rf.feature_importances_, index=X_rf.columns).sort_values(ascending=False)

# 5. Permutation Importance
perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42)
perm_importances = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)

# 6. Combine into one DataFrame
importance_df = pd.DataFrame({
    'RandomForest Importance': rf_importances,
    'Permutation Importance': perm_importances
}).sort_values(by='RandomForest Importance', ascending=True)

# 7. Plot
importance_df.plot(kind='barh', figsize=(10, 6))
plt.title("Feature Importance: Random Forest vs. Permutation")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()

# 8. Display numeric table
importance_df = importance_df.sort_values(by='RandomForest Importance', ascending=False)
display(importance_df)

# 9. Model Evaluation
from sklearn.metrics import r2_score, mean_squared_error

y_pred = rf.predict(X_test)

# Evaluation Metrics
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print(f"R^2 Score: {r2:.4f}")
print(f"Mean Squared Error: {mse:.4f}")


NameError: name 'X_full' is not defined

# Random Forest Optimised

In [10]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
import matplotlib.pyplot as plt

# 1. Drop 'const' and define input/output
X_rf = X_full.drop(columns='const')
y_rf = y

# 2. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X_rf, y_rf, test_size=0.3, random_state=42)

# 3. Define Random Forest Model
rf = RandomForestRegressor(random_state=42)

# 4. Define Hyperparameter Grid for Random Search (updated to remove 'auto')
param_grid = {
    'n_estimators': [100, 300, 500, 700, 1000],
    'max_depth': [5, 10, 15, 20, 30, None],
    'min_samples_split': [2, 5, 10, 15, 20],
    'min_samples_leaf': [1, 2, 4, 6, 8],
    'max_features': [1.0, 'sqrt', 'log2', 0.3, 0.5, 0.75],
    'bootstrap': [True, False]
}

# 5. Initialize RandomizedSearchCV
rf_random = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_grid,
    n_iter=200,  # Increased for better tuning
    scoring='r2',  # Can also try 'neg_mean_squared_error'
    cv=10,  # Increased cross-validation for robustness
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# 6. Fit the model with RandomizedSearchCV
rf_random.fit(X_train, y_train)

# 7. Extract the Best Hyperparameters & Score
best_params = rf_random.best_params_
best_score = rf_random.best_score_

print("Best Hyperparameters:", best_params)
print(f"Best R² Score from Cross-Validation: {best_score:.4f}")

# 8. Train the Final Model using Best Hyperparameters
best_rf = RandomForestRegressor(**best_params, random_state=42)
best_rf.fit(X_train, y_train)

# 9. Make Predictions on Training and Test Data
y_train_pred = best_rf.predict(X_train)
y_test_pred = best_rf.predict(X_test)

# 10. Evaluate the Tuned Model (Train & Test)
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)

# Print Results
print(f"\nTraining Performance:")
print(f"Train R² Score: {train_r2:.4f}")
print(f"Train Mean Squared Error: {train_mse:.4f}")

print(f"\nTest Performance:")
print(f"Test R² Score: {test_r2:.4f}")
print(f"Test Mean Squared Error: {test_mse:.4f}")

# === Debugging Check Before Step 11 ===

# Checking the number of features before model training
print(f"\nNumber of features in X_rf: {X_rf.shape[1]}")
print(f"Number of feature importances from Random Forest: {len(best_rf.feature_importances_)}")

# Make sure all columns match correctly
if X_rf.shape[1] == len(best_rf.feature_importances_):
    # 11. Feature Importance Plot
    rf_importances = pd.Series(best_rf.feature_importances_, index=X_rf.columns).sort_values(ascending=False)

    plt.figure(figsize=(10, 6))
    rf_importances.plot(kind='barh')
    plt.title("Feature Importance (Tuned Random Forest)")
    plt.xlabel("Importance Score")
    plt.tight_layout()
    plt.show()

    # 12. Display Feature Importances in DataFrame
    importance_df = pd.DataFrame({
        'Feature': X_rf.columns,
        'Importance': best_rf.feature_importances_
    }).sort_values(by='Importance', ascending=False)

    display(importance_df)
else:
    print("⚠️ Feature mismatch detected! Ensure preprocessing steps are consistent.")


NameError: name 'X_full' is not defined

# OLS vs Random Forest

In [11]:
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error


# === Comparison of Performance Metrics ===
ols_pred = model.predict(X_full)
ols_r2 = r2_score(y, ols_pred)
ols_mse = mean_squared_error(y, ols_pred)
ols_mae = mean_absolute_error(y, ols_pred)

rf_train_r2 = train_r2
rf_test_r2 = test_r2
rf_train_mse = train_mse
rf_test_mse = test_mse
rf_mae = mean_absolute_error(y_test, y_test_pred)

print(f"\n{'Metric':<25} {'OLS':<10} {'Random Forest (Test)':<10}")
print(f"{'-'*50}")
print(f"{'R² Score':<25} {ols_r2:.4f}      {rf_test_r2:.4f}")
print(f"{'MSE':<25} {ols_mse:.4f}      {rf_test_mse:.4f}")
print(f"{'MAE':<25} {ols_mae:.4f}      {rf_mae:.4f}")

# === Plotting Feature Importance for OLS and Random Forest ===
ols_importances = pd.Series(model.params.values, index=model.params.index).sort_values(ascending=False)
rf_importances = pd.Series(best_rf.feature_importances_, index=X_rf.columns).sort_values(ascending=False)

plt.figure(figsize=(15, 6))
plt.subplot(1, 2, 1)
ols_importances.plot(kind='barh')
plt.title("OLS Regression Feature Importance")
plt.xlabel("Coefficient Value")

plt.subplot(1, 2, 2)
rf_importances.plot(kind='barh')
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance Score")

plt.tight_layout()
plt.show()

# === Comparing Top 20 Features from Both Models ===
importance_comparison = pd.DataFrame({
    'OLS Coefficients': ols_importances,
    'RF Importance': rf_importances
}).sort_values(by='RF Importance', ascending=False)

display(importance_comparison.head(20))

# === Visual Comparison of Predictions ===
plt.figure(figsize=(15, 6))

plt.subplot(1, 2, 1)
plt.scatter(y, ols_pred, alpha=0.6)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
plt.title('OLS: True vs Predicted')
plt.xlabel('True Values')
plt.ylabel('Predicted Values')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.scatter(y_test, y_test_pred, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.title('Random Forest: True vs Predicted')
plt.xlabel('True Values')
plt.ylabel('Predicted Values')
plt.grid(True)

plt.tight_layout()
plt.show()

NameError: name 'model' is not defined

# QUANTILE REGRESSION

In [12]:
# STEP 8: FIT QUANTILE REGRESSION MODEL (for Quantile 0.25)
quantile = 0.5  # Adjust this value for different quantiles (0.25, 0.5, 0.75, etc.)
model = QuantReg(y, X_full).fit(q=quantile)

# Display model summary
display(model.summary())

# STEP 9: Calculate Pseudo R² (for comparison with OLS and other quantiles)
y_pred = model.predict(X_full)
S_fitted = np.sum(np.abs(y - y_pred))
S_baseline = np.sum(np.abs(y - np.mean(y)))
pseudo_r2 = 1 - (S_fitted / S_baseline)
display(f"\nPseudo R² for Quantile {quantile}: {pseudo_r2 * 100:.2f}%")

NameError: name 'y' is not defined

# SHAP Analysis

In [ ]:
# Step 1: Clean feature set
X_final_cleaned = X_full.drop(columns=[col for col in all_exclusions if col in X_full.columns], errors='ignore')

# Step 2: Define target (already standardized and cleaned)
y_final = y

# Step 3: Fit Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_final_cleaned, y_final)

# Step 4: SHAP analysis
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_final_cleaned)

# Step 6: SHAP mean importance table
shap_df = pd.DataFrame(shap_values, columns=X_final_cleaned.columns)
shap_mean_importance = shap_df.abs().mean().sort_values(ascending=False)

shap_importance_df = pd.DataFrame({
    'Feature': shap_mean_importance.index,
    'Mean |SHAP Value|': shap_mean_importance.values
})

# Step 7: Show importance table
display(shap_importance_df)

# Step 5: SHAP summary plot
shap.initjs()
shap.summary_plot(shap_values, X_final_cleaned)
plt.show()

# Step 6: SHAP Summary Plot - Beeswarm Plot
plt.figure()
shap.summary_plot(shap_values, X_final_cleaned, plot_type="dot", show=False)
plt.title("SHAP Summary Plot - Beeswarm Plot")
plt.tight_layout()
plt.savefig("SHAP_Summary_Beeswarm_Plot.png", dpi=300)
plt.show()
plt.close()

# Step 9: Generate Dependence Plots for Key Features
import re

key_features = ['Starting_Size', 'PKD_1_D', 'P/F_P']
for feature in key_features:
    plt.figure()
    shap.dependence_plot(feature, shap_values, X_final_cleaned, show=False)
    plt.title(f"SHAP Dependence Plot - {feature}")
    plt.tight_layout()
    
    # Replace special characters with an underscore
    safe_feature_name = re.sub(r'[^\w\-_\. ]', '_', feature)
    plt.savefig(f"SHAP_Dependence_{safe_feature_name}.png", dpi=300)
    plt.show()
    plt.close()

# Lasso Analysis

In [ ]:
# STEP 1: Use cleaned modeling dataset from earlier steps
df_lasso = df_model.copy()  # Already Year == 2023 and excludes unwanted columns

# STEP 2: Separate features and target
X_raw = df_lasso.drop(columns=[target_variable])
y = df_lasso[target_variable]

# STEP 3: Split numeric and categorical
numeric_cols = X_raw.select_dtypes(include=[np.number]).columns
categorical_cols = X_raw.select_dtypes(include=['object', 'category']).columns

# STEP 4: Clean and scale numeric
X_num = X_raw[numeric_cols].replace([np.inf, -np.inf], np.nan).dropna()
X_scaled = pd.DataFrame(StandardScaler().fit_transform(X_num), columns=numeric_cols, index=X_num.index)

# STEP 5: Encode categorical variables
X_cat = pd.get_dummies(X_raw.loc[X_scaled.index, categorical_cols], drop_first=True)

# STEP 6: Combine numeric + categorical
X = pd.concat([X_scaled, X_cat], axis=1).replace([np.inf, -np.inf], np.nan).dropna()
y = y.loc[X.index]  # Align target

# STEP 7: Run LassoCV
lasso_cv = LassoCV(cv=5, max_iter=20000, random_state=42)
lasso_cv.fit(X, y)

# STEP 8: Display selected features
lasso_coef = pd.Series(lasso_cv.coef_, index=X.columns)
selected_features = lasso_coef[lasso_coef != 0].sort_values(ascending=False)

display(HTML("<strong>✅ Lasso-selected explanatory variables:</strong>"))
display(selected_features)